<div dir="rtl">
<h1>پنج نمونه، سه Batch و یک آخرِ کوچک‌تر</h1>
<p>درس 24 از 76 · یک نمونه چطور به Batch تبدیل می‌شود؟ · <code dir="ltr">20-loader</code></p>
<p><a target="_self" href="http://127.0.0.1:8000/part-03/chapter-04/20-loader.html">📖 بازگشت به همین درس</a></p>
<p>Dataset را خودتان بسازید و پوشش همهٔ نمونه‌ها را بررسی کنید.</p><p>پیش‌نیاز: 14-index-device و19-network؛ Dataset و DataLoader درس جاری.</p>
<p>این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو Cell با برچسب TODO را خودتان کامل کنید. پیام INCOMPLETE یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p>از بالا به پایین اجرا کنید. پس از تغییر هر تابع، Cell آن و سپس Cell آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code>Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import os
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl">
<h2>قبل از اجرا، پیش‌بینی کنید</h2>
<p>پنج نمونه با batch_size=2 چند Batch می‌سازند؟ آخرین نمونه با drop_last=True چه می‌شود؟</p>
</div>

<div dir="rtl"><p>پیش‌بینی من: …</p></div>

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
print("Indices to visit:", list(range(5)))

<div dir="rtl">
<h2>این بار شما کد بنویسید</h2>
<p>تابع make_dataset(n) یک Dataset برگرداند: ورودی نمونهٔ i برابر Tensor اعشاری [i] و هدف، Tensor صحیحِ تکی i%2 باشد. __len__ باید n باشد و اندیس بیرون [0,n) خطای IndexError بدهد.</p>
</div>

In [ ]:
def make_dataset(n):
    # TODO: return a Dataset implementing __len__ and __getitem__
    return None

In [ ]:
def test_exercise():
    result = make_dataset(5)
    if result is None:
        return False
    assert len(result) == 5
    features, target = result[3]
    assert features.tolist() == [3.] and features.dtype == torch.float32
    assert target.shape == () and target.dtype == torch.long and target.item() == 1
    batches = list(DataLoader(result, batch_size=2, shuffle=False, num_workers=0))
    assert [len(features) for features, _ in batches] == [2, 2, 1]
    assert torch.cat([features[:, 0] for features, _ in batches]).tolist() == [0, 1, 2, 3, 4]
    try:
        result[5]
    except IndexError:
        pass
    else:
        raise AssertionError("Dataset index must be bounded")
    return True

exercise_complete = test_exercise()
print('PASS' if exercise_complete else 'INCOMPLETE: implement the TODO and rerun')

<div dir="rtl">
<h2>فقط یک عامل را تغییر دهید</h2>
<p>فقط drop_last را عوض کنید. برای این مقایسه از فهرست Tensor آماده استفاده می‌کنیم تا به پیاده‌سازی تمرین وابسته نباشد.</p>
</div>

In [ ]:
samples = [torch.tensor([i]) for i in range(5)]
for drop in [False, True]:
    loader = DataLoader(samples, batch_size=2, shuffle=False, drop_last=drop, num_workers=0)
    print("drop_last:", drop, "visited:", [batch.flatten().tolist() for batch in loader])

<div dir="rtl">
<h2>خرابی را پیدا کنید</h2>
<p>حلقهٔ خراب با floor division آخرین Batch را فراموش می‌کند. تابع batch_ranges(n,size) جفت‌های (start,stop) بسازد تا هر اندیس دقیقاً یک بار پوشش داده شود؛ size مثبت است.</p>
</div>

In [ ]:
wrong_ranges = [(i*2, (i+1)*2) for i in range(5//2)]
print("Broken ranges:", wrong_ranges)
assert wrong_ranges[-1][1] == 4

<div dir="rtl">
<h2>اصلاح را خودتان بنویسید</h2>
<p>علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def batch_ranges(n, size):
    # TODO: keep the incomplete final batch
    return None

In [ ]:
def test_repair():
    result = batch_ranges(5, 2)
    if result is None:
        return False
    assert result == [(0, 2), (2, 4), (4, 5)]
    assert batch_ranges(4, 2) == [(0, 2), (2, 4)]
    assert batch_ranges(1, 8) == [(0, 1)]
    return True

repair_complete = test_repair()
print('PASS' if repair_complete else 'INCOMPLETE: implement the TODO and rerun')

<div dir="rtl">
<h2>در Mini-GPT کجا به کار می‌آید؟</h2>
<p>ارزیابی در mini_gpt/evaluate.py همهٔ نمونه‌های NextTokenDataset را با DataLoader می‌بیند. آموزش تصادفی پروژه قرارداد دیگری دارد؛ هر تعداد Step الزاماً یک Epoch نیست.</p>
</div>

<div dir="rtl">
<h2>با زبان خودتان توضیح دهید</h2>
<p>اگر آخرین Batch کوچک‌تر باشد، چرا نباید Loss آن را هم‌وزن یک Batch بزرگ میانگین بگیرید؟</p>
</div>
<div dir="rtl"><p>پیش‌بینی و مشاهدهٔ من: …</p><p>علت خرابی و اصلاح من: …</p></div>

<div dir="rtl"><p><a target="_self" href="http://127.0.0.1:8000/part-03/chapter-04/20-loader.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/20-loader.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>